In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
from matplotlib.ticker import FuncFormatter
from matplotlib.gridspec import GridSpec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

from matplotlib.colors import LogNorm
# from scipy.interpolate import interp1d  
from scipy import stats
from matplotlib.ticker import LogLocator
from matplotlib.backends.backend_pdf import PdfPages

import pandas as pd

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
#Loading Data Functions
def GetData(tRange=[12,13], pValues=np.arange(20_000_000)):
    t1 = calculate_timestep(tRange[0])
    t2 = calculate_timestep(tRange[1])

    varNames = ['Z','Y','X'] + ['QCQI','QV','W']
    dtypes = {'Z': np.int16, 
              'Y': np.int16, 
              'X': np.int16, 
              'QCQI': np.float32, 
              'QV': np.float32,
               'W': np.float32}
    
    Nt = t2 - t1 + 1
    Np = len(pValues)
    dataDictionary = {var: np.zeros((Nt, Np), dtype=dtypes[var]) for var in varNames}
    dataDictionary['tSteps'] = np.arange(t1, t2+1)
    dataDictionary['pValues'] = pValues
    
    for i, t in enumerate(tqdm(dataDictionary['tSteps'])):
        for varName in varNames:
            raw = CallLagrangianArray(ModelData, DataManager, ModelData.timeStrings[t], varName)[pValues]
            dataDictionary[varName][i, :] = np.round(raw) if varName in ('Z','Y','X') else raw
            
    return dataDictionary

def calculate_timestep(time_hr=12):
    return np.abs(ModelData.time_hrs-time_hr).argmin()

def LimitTrackedArraysRows(trackedArrays, limit=None): #limit=(0,70000)
    if limit is None:
        return trackedArrays
    for parcelType in trackedArrays:
        for parcelDepth in trackedArrays[parcelType]:
            trackedArrays[parcelType][parcelDepth] \
            = trackedArrays[parcelType][parcelDepth][limit[0]:limit[1], :]
    return trackedArrays

In [ ]:
# Functions for Calculating Lagrangian version of CloudType (from below)
def CalculateLagrangianCloudType(dataDictionary,tRange=[12,13]):
    t1 = calculate_timestep(tRange[0])
    t2 = calculate_timestep(tRange[1])
    Z=dataDictionary['Z'];Y=dataDictionary['Y'];X=dataDictionary['X']
    cloudTypeMap_Lagrangian = np.full_like(Z, -1, dtype=np.int8)
    
    for (t_rel,t) in enumerate(tqdm(range(t1,t2+1))):
        [w,rc,ri]=LoadData(t=t)
        A = GetCloudArray(rc,ri,w)
        [L,counts]=GetLabelArray(A)
        cloudTypes=ClassifyClouds(L)
        cloudTypeMap = GetTypeCodeMap(L,cloudTypes)
        cloudTypeMap_Lagrangian[t_rel] = cloudTypeMap[Z[t_rel], Y[t_rel], X[t_rel]]

    return cloudTypeMap_Lagrangian
        
        

# Functions for Calculating CloudType (from in /2_Variable_Calculation/2_CalculateMoreVariables)
################################################################################################################################
################################################################################################################################
def GetVarNames():
    return ['winterp','qc','qi']

def GetInputVariables(inputDataDirectory, timeString, varNames):
    inputDictionary = {varName: CallVariable(ModelData, DataManager, timeString, varName) for varName in varNames}
    return inputDictionary

def LoadData(t,verbose=False):
    if verbose: print(f'Loading data at {ModelData.time_hrs[t]} LT')
    varNames = GetVarNames()
    timeString = ModelData.timeStrings[t]
    inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString, varNames)
    [w,rc,ri] = (inputDictionary[k] for k in varNames)
    return [w,rc,ri]

#Calculation Functions

#---Binary Array Calcaultion Function---
def GetCloudArray(rc,ri,w):
    """
    Calculates cloudy binary threshold array
    """
    condition1=(rc+ri>1e-5)
    condition2=(w>0)
    A = condition1&condition2
    return A

#---3D Connected-Component Labeling Function---
from scipy.ndimage import label, generate_binary_structure
from scipy.ndimage import label, generate_binary_structure, maximum_filter
from skimage.segmentation import watershed

def GetLabelArray(A, periodic_yx=(True, False), minCount=3):  # minCount=9 Champouillon et al. 2023
    """
    Identify 3D cloud objects via connected-component labeling, following
    the object identification method of:

        Champouillon, A., C. Rio, and F. Couvreux, 2023: Simulating the
        Transition from Shallow to Deep Convection across Scales: The Role
        of Congestus Clouds. J. Atmos. Sci., 80, 2989-3005,
        https://doi.org/10.1175/JAS-D-23-0027.1

    Champouillon et al. define a cloud as "a set of contiguous cloudy
    cells" and discard objects smaller than 9 cells (their Sec. 4a),
    accounting for domain periodicity at the lateral boundaries. This
    function reproduces that definition using 26-connectivity (face +
    edge + corner adjacency; not specified in the paper) and periodic
    padding along the axes given by `periodic_yx`.
    """
    [periodic_y, periodic_x] = periodic_yx
    structure = generate_binary_structure(3, 3)  # 26-connectivity

    pad_y = (1, 1) if periodic_y else (0, 0)
    pad_x = (1, 1) if periodic_x else (0, 0)

    if periodic_y or periodic_x:
        A_padded = np.pad(A, pad_width=((0, 0), pad_y, pad_x), mode='wrap')
        L_padded, n = label(A_padded, structure=structure)

        y_slice = slice(1, -1) if periodic_y else slice(None)
        x_slice = slice(1, -1) if periodic_x else slice(None)
        L = L_padded[:, y_slice, x_slice]
    else:
        L, n = label(A, structure=structure)

    counts = np.bincount(L.ravel())
    small = counts < minCount
    small[0] = False
    L = np.where(small[L], 0, L)

    # relabel to close gaps left by the size filter, so every remaining
    # id from 1..L.max() corresponds to a real, nonempty object
    L, n_final = label(L > 0, structure=structure)
    counts = np.bincount(L.ravel())   # recompute counts to match the new numbering
    return [L, counts]

#---Cloud Type Identification Algorithm---
from scipy import ndimage
import pandas as pd

def GetTypeCodeMap(L,cloudTypes):
    """
    Build a spatial field the same shape as L, where each cell holds an
    integer code for its object's classified cloud type:
        1 = cumulus, 2 = congestus, 3 = cumulonimbus, 4 = overshooting,
        -1 = unclassified (or any other/unrecognized type)
    Background (L == 0) also gets -1.
    """
    type_code = {
        'cumulus': 1,
        'congestus': 2,
        'cumulonimbus': 3,
        'overshooting': 4,
    }

    max_label = int(L.max())
    label_to_code = np.full(max_label + 1, -1)  # default: unclassified/background

    # pull label ids and their corresponding type codes out of the DataFrame
    labels_arr = cloudTypes['label'].values.astype(int)
    codes_arr = cloudTypes['cloud_type'].map(type_code).fillna(-1).values.astype(int)
    label_to_code[labels_arr] = codes_arr
    cloudTypeMap=label_to_code[L]
    return cloudTypeMap

def ClassifyClouds(L, zh=ModelData.zh, scheme='kumar_champouillon_hybrid'):
    n_labels = L.max()
    if n_labels == 0:
        return pd.DataFrame(columns=['label', 'top_km', 'base_km', 'depth_km', 'cloud_type'])

    # 3D height field where every cell holds its z-level's height (zh[k]), 
    # so it can be compared cell-by-cell against the labeled array L
    Z = np.broadcast_to(zh[:, None, None], L.shape)
    idx = np.arange(1, n_labels + 1)

    # max height among each label's cells -> cloud-top height per object
    tops = ndimage.maximum(Z, labels=L, index=idx)
    bases = ndimage.minimum(Z, labels=L, index=idx)
    depths = tops - bases

    cloud_types = [classify_object(t, d, b, scheme=scheme) for t, d, b in zip(tops, depths, bases)]  # <-- added b

    cloudTypes = pd.DataFrame({
        'label': idx, 'top_km': tops, 'base_km': bases,
        'depth_km': depths, 'cloud_type': cloud_types,
    })
    return cloudTypes

def classify_object(top_km, depth_km, base_km, scheme):
    if scheme not in SCHEMES:
        raise ValueError(f"scheme must be one of {list(SCHEMES)}, got '{scheme}'")

    for cloud_type, ranges in SCHEMES[scheme].items():
        top_ok = check_in_range(top_km, ranges['top'], inclusive=(False, True))    # (min, max]
        depth_ok = check_in_range(depth_km, ranges['depth'], inclusive=(True, True))  # [min, max]
        base_ok = check_in_range(base_km, ranges.get('base', (None, None)), inclusive=(True, True))  # [min, max]
        if top_ok and depth_ok and base_ok:
            return cloud_type
    return 'unclassified'
    
def check_in_range(value, bounds, inclusive=(False, True)):
    """checks if value is in bounds"""
    vmin, vmax = bounds
    min_incl, max_incl = inclusive

    ok_min = vmin is None or (value >= vmin if min_incl else value > vmin)
    ok_max = vmax is None or (value <= vmax if max_incl else value < vmax)
    return ok_min and ok_max
    
# --- Threshold Dictionaries (units: km, matching ModelData.zh) ---
KUMAR_SCHEME = {
    # Kumar et al. (2013)
    # Kumar, V. V., C. Jakob, A. Protat, P. T. May, and L. Davies, 2013: The
    # Four Cumulus Cloud Modes and Their Progression During Rainfall Events:
    # A C-Band Polarimetric Radar Perspective. J. Geophys. Res. Atmos., 118,
    # 8375-8389, https://doi.org/10.1002/jgrd.50640
    'cumulus':      {'top': (1.0, 3.0),   'depth': (None, None)},
    'congestus':    {'top': (3.0, 6.5),   'depth': (None, None)},
    'cumulonimbus': {'top': (6.5, 15.0),  'depth': (None, None)},
    'overshooting': {'top': (15.0, None), 'depth': (None, None)},
}

CHAMPOUILLON_SCHEME = {
    # Champouillon et al. (2023)
    # Champouillon, A., C. Rio, and F. Couvreux, 2023: Simulating the
    # Transition from Shallow to Deep Convection across Scales: The Role
    # of Congestus Clouds. J. Atmos. Sci., 80, 2989-3005,
    # https://doi.org/10.1175/JAS-D-23-0027.1
    'cumulus':      {'top': (None, 3.0), 'depth': (None, None)},
    'congestus':    {'top': (3.0, 6.0),  'depth': (0.5, None)},
    'cumulonimbus': {'top': (6.0, None), 'depth': (0.5, None)},
    'others':       {'top': (3.0, None), 'depth': (None, 0.5)},
}


KUMAR_CHAMPOUILLON_HYBRID_SCHEME = {
    # Kumar et al. (2013)
    # Champouillon et al. (2023)
    'cumulus':      {'top': (1.0, 3.0),   'depth': (None, None), 'base': (0.5, 3.0)},
    'congestus':    {'top': (3.0, 6.5),   'depth': (0.5, None),  'base': (0.5, 3.0)},
    'cumulonimbus': {'top': (6.5, 15.0),  'depth': (0.5, None),  'base': (0.5, 3.0)},
    'overshooting': {'top': (15.0, None), 'depth': (0.5, None),  'base': (0.5, 3.0)},
}

SCHEMES = {
    'kumar': KUMAR_SCHEME,
    'champouillon': CHAMPOUILLON_SCHEME,
    'kumar_champouillon_hybrid': KUMAR_CHAMPOUILLON_HYBRID_SCHEME,
}
################################################################################################################################
################################################################################################################################

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
#Track Ascending Surface Parcels

def DetectUpdraftEvents(dataDictionary, wThresh=0.1,cloudThreshold=1e-5, 
                        minAscentTime_mins=20,minCloudTime_mins=5, 
                        minHeightGain=1.0, surfaceZLimit=0.5):
    W  = dataDictionary['W']
    QC = dataDictionary['QCQI']
    Z  = ModelData.zh[dataDictionary['Z']]
    Nt = Z.shape[0]
    
    ascentWindow = MinutesToTimesteps(minAscentTime_mins)
    minCloudSteps = MinutesToTimesteps(minCloudTime_mins)
    
    sustainedUpdraft = RollingAllTrueInWindow(W > wThresh, ascentWindow,
                                              switch='every_step_true') # w>thresh whole window
    cloudSteps = RollingAllTrueInWindow(QC > cloudThreshold, ascentWindow,
                                        switch='true_count') # cloudy for enough of it
    #net change in a parcel's height between the first and last timestep of 
    #the ascentWindow-length span starting at T
    heightGain = Z[ascentWindow-1:] - Z[:Nt-ascentWindow+1]

    n = sustainedUpdraft.shape[0] # target length for the shifted checks below
    
    # only parcels that initiate below 0.5 km
    wasSurfaceParcel = np.zeros_like(sustainedUpdraft)
    wasSurfaceParcel[1:] = Z[:n-1] <= surfaceZLimit

    # make sure parcel undergoes acceleration (w failed threshold the step before)
    notUpdraftBefore = np.zeros_like(sustainedUpdraft)
    notUpdraftBefore[1:] = ~(W[:n-1] > wThresh)
    
    #returns True at the beginning time of the continued ascentWindow-time ascent if all thresholds
    #are passed for ascentWindow times after that first time
    isUpdraftEvent_ascentRelative = ( 
        sustainedUpdraft
        & (cloudSteps >= minCloudSteps)
        & wasSurfaceParcel
        & notUpdraftBefore
        & (heightGain >= minHeightGain)
    ) #shape (Nt-ascentWindow, Np); True at t = event starting at t

    #padding isUpdraftEvent_ascentRelative with False
    isUpdraftEvent = PadToFullNt(isUpdraftEvent_ascentRelative, Nt=dataDictionary['Z'].shape[0])
    return isUpdraftEvent

def MinutesToTimesteps(minutes):
    """Converts a duration in minutes to a number of model timesteps."""
    return max(1, round((minutes * 60) / ModelData.dt))

def RollingAllTrueInWindow(cond, window,
                           switch='true_count'):
    """
    True at row T if cond is True for every step in T ... T + window-1.
    Comments:
    c: running total of True's from the start up through each row.
    (c[window:] - c[:-window]): count of True's inside each window-length
    (c[window:] - c[:-window]) == window: turns that count into a yes/no.
    """
    c = np.cumsum(np.vstack([np.zeros((1, cond.shape[1])), cond]), axis=0) #counts number of True

    if switch=='true_count':
        return c[window:] - c[:-window]
    elif switch=='every_step_true':
        return (c[window:] - c[:-window]) == window

def PadToFullNt(eventArray, Nt, fillValue=False):
    """
    Extends a windowed/rolled array back up to the full Nt timesteps
    by appending rows of `fillValue` (always keep set to False) at the end.
    """
    Np = eventArray.shape[1]
    nMissing = Nt - eventArray.shape[0]
    pad = np.full((nMissing, Np), fillValue, dtype=eventArray.dtype)
    return np.vstack([eventArray, pad])

####################################################################################
#Get Full-Duration Ascent Mask
def GetAscentDurationMask(dataDictionary, isUpdraftEvent, wThresh=0.1,
                           capAtCloudExit=True, cloudThreshold=1e-5):
    W = dataDictionary['W']
    Nt, Np = W.shape

    rawUpdraft = W > wThresh
    runStart, runEnd, runParcel = FindRuns(rawUpdraft)

    eventT, eventP = np.where(isUpdraftEvent)

    eventDF = pd.DataFrame({'t': eventT, 'p': eventP})
    runDF = pd.DataFrame({'t': runStart, 'end': runEnd, 'p': runParcel})
    matched = eventDF.merge(runDF, on=['t', 'p'], how='inner')

    if capAtCloudExit:                                                      
        matched = matched.copy()
        matched['end'] = GetCloudCappedRunEnd(dataDictionary, matched, cloudThreshold)

    isAscending = BuildFullDurationMask(matched['t'].values,
                                        matched['end'].values,
                                        matched['p'].values, Nt, Np)
    return isAscending

def FindRuns(condition):
    Nt, Np = condition.shape
    padded = np.zeros((Nt + 2, Np), dtype=np.int8)
    padded[1:-1, :] = condition
    d = np.diff(padded, axis=0)
    startRows, startCols = np.where(d == 1) #Start: transition to updraft
    endRows, endCols = np.where(d == -1) #End: transition away from updraft
    
    # indices that would sort the start-events by parcel, then by time
    orderS = np.lexsort((startRows, startCols))
    # indices that would sort the end-events by parcel, then by time
    orderE = np.lexsort((endRows, endCols))
    
    #Reorders startRows/startCols into (parcel, time) order
    #so each parcel's runs are grouped together chronologically
    startRows, startCols = startRows[orderS], startCols[orderS]
    endRows = endRows[orderE]
    return startRows, endRows, startCols 

def BuildFullDurationMask(startIdx, endIdx, parcelIdx, Nt, Np):
    delta = np.zeros((Nt + 1, Np), dtype=np.int8) # one extra row of padding at end
    np.add.at(delta, (startIdx, parcelIdx), 1) # mark +1 where each event starts
    np.add.at(delta, (np.clip(endIdx, 0, Nt), parcelIdx), -1) # mark -1 where each event ends
    # running total ("on" since start, "off" after end) and drop padding row
    return np.cumsum(delta[:-1], axis=0) > 0 

def GetCloudCappedRunEnd(dataDictionary, matched, cloudThreshold=1e-5):
    """
    For each confirmed ascending run (matched['t'] to matched['end']),
    caps the end at the first moment the parcel exits cloud (QCQI drops
    back below cloudThreshold) AFTER it first became cloudy -- so an
    early, pre-condensation cloud-free stretch doesn't immediately end
    the run, but exiting cloud after entering it does.
    """
    QC = dataDictionary['QCQI']
    starts = matched['t'].values
    ends = matched['end'].values
    parcels = matched['p'].values
    cappedEnd = ends.copy()

    for i in range(len(starts)):
        s, e, p = starts[i], ends[i], parcels[i]
        isCloudy = QC[s:e, p] > cloudThreshold

        if not isCloudy.any():
            continue   # never became cloudy in this run -- keep original w-based end

        firstCloudyIdx = np.argmax(isCloudy)          # first True (first condensation point)
        afterCloudy = isCloudy[firstCloudyIdx:]
        exitedCloud = ~afterCloudy
        if exitedCloud.any():
            exitIdx = np.argmax(exitedCloud)           # first False AFTER first becoming cloudy
            cappedEnd[i] = s + firstCloudyIdx + exitIdx
        # else: stays cloudy all the way through the original w-based end -- leave as is

    return cappedEnd

####################################################################################

def PlotAscentValidation(dataDictionary, isAscending, p, wThresh=0.1, cloudThreshold=1e-5*1e3):
    W = dataDictionary['W'][:, p]
    QC = dataDictionary['QCQI'][:, p]*1e3
    tSteps = dataDictionary['tSteps']
    timeOfDay = ModelData.time_hrs[tSteps]

    fig, ax1 = plt.subplots(figsize=(10, 4))

    ax1.plot(timeOfDay, W, color='black', lw=1.5, label='w')
    ax1.axhline(wThresh, color='gray', ls='--', lw=1, label=f'wThresh={wThresh}')
    ax1.set_xlabel('time of day')
    ax1.set_ylabel('w (m/s)')
    ax1.xaxis.set_major_formatter(FuncFormatter(HoursToHHMM))

    mask = isAscending[:, p]

    # duration of the ascending region, from the actual masked timesteps
    if mask.any():
        durMin = round(mask.sum() * ModelData.dt / 60)
        ascentLabel = f'isAscending ({durMin} min)'
    else:
        ascentLabel = 'isAscending (none)'

    ax1.fill_between(timeOfDay, ax1.get_ylim()[0], ax1.get_ylim()[1], where=mask,
                      color='tab:orange', alpha=0.15, step='mid', label=ascentLabel)

    ax2 = ax1.twinx()
    ax2.plot(timeOfDay, QC, color='blue', lw=1.5, label='QCQI')
    ax2.axhline(cloudThreshold, color='blue', ls=':', lw=1, alpha=0.6,
                label=f'cloudThreshold={cloudThreshold}')
    ax2.set_ylabel('QCQI (cloud water+ice)', color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

    ax1.set_title(f'Parcel {p} — w and QCQI profile vs detected ascent')
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

def HoursToHHMM(hrs, pos=None):
    h = int(hrs)
    m = int(round((hrs - h) * 60))
    if m == 60:
        h += 1
        m = 0
    return f"{h:02d}:{m:02d}"

In [ ]:
#Find Detraining Parcels at MidLevels
def DetectDetrainment(dataDictionary, zLimit=(1,6.5), cloudThreshold=1e-5):
    zLevel = ModelData.zh[dataDictionary['Z']]
    QC = dataDictionary['QCQI']  # using qcqi for now

    inLayer = (zLevel >= zLimit[0]) & (zLevel <= zLimit[1])
    isCloudy = QC > cloudThreshold
    isEnvironment = ~isCloudy

    wasCloudy = np.zeros_like(isCloudy)
    wasCloudy[1:] = isCloudy[:-1]  # shift cloudy at t-1 boolean to time t

    movedGridbox = DetectGridboxCross(dataDictionary)

    isDetrain = wasCloudy & isEnvironment & inLayer & movedGridbox
    return isDetrain

def DetectGridboxCross(dataDictionary):
    X = dataDictionary['X']
    Y = dataDictionary['Y']

    xChanged = np.zeros_like(X, dtype=bool)
    yChanged = np.zeros_like(Y, dtype=bool)
    xChanged[1:] = X[1:] != X[:-1]
    yChanged[1:] = Y[1:] != Y[:-1]

    return xChanged | yChanged

In [ ]:
#Find Ascending-Parcel Intersections with Detrainment Events

def FindIntersections(dataDictionary, isAscending, isDetrain, 
                      windowMin=30, cellRadius_km=1):
    dims = GetGridDims()
    cellId = AssignCellId(dataDictionary) #get cell ids
    Nt, Np = isAscending.shape
    windowSteps = MinutesToTimesteps(windowMin)
    cellRadius = CellRadiusKmToIndex(cellRadius_km, ModelData.dx) #radius to check neighbors

    # detrainment events: where and when each one happened
    dT, dP = np.where(isDetrain)
    dCell = cellId[dT, dP]
    dIx, dIy, dIz = InverseCellId(dCell) #get Z,Y,X ids back

    # generate the neighborhood of cell IDs around each event
    [neighborCellIds, numNeighbors] = GetNeighborCellIds(dIx, dIy, dIz, dims, radius=cellRadius)
    # repeat time/parcel arrays so their length matches the expanded neighbor list
    dT_expanded = np.repeat(dT, numNeighbors)
    dP_expanded = np.repeat(dP, numNeighbors)
    neighborCellIds_flat = neighborCellIds.ravel()

    # stretch each event forward so it stays "active" for windowMin minutes
    offsets = np.arange(windowSteps) # 0, 1, 2, ..., windowSteps-1
    expT = (dT_expanded[:, None] + offsets[None, :]).ravel() # every (t0 + offset) for every event, flattened
    expCell = np.repeat(neighborCellIds_flat, windowSteps) # same gridbox, repeated once per offset
    expDetrainParcel = np.repeat(dP_expanded, windowSteps) # same source parcel, repeated once per offset
    valid = expT < Nt # drop times that run past the end of loaded data
    detrainDF = pd.DataFrame({
        't': expT[valid], 'cellId': expCell[valid], 'detrain_parcel': expDetrainParcel[valid]
    }) #dataframe listing time and location of valid detrainment for each parcel

    # ascending parcels: where each one is at every timestep
    aT, aP = np.where(isAscending)
    aCell = cellId[aT, aP]
    ascendDF = pd.DataFrame({'t': aT, 'cellId': aCell, 'ascend_parcel': aP})

    #keep only rows where an ascending parcel's (time, gridbox) matches entrainment parcel's
    matches = ascendDF.merge(detrainDF, on=['t', 'cellId'], how='inner')
    # drop ascending parcels that detrained
    matches = matches[matches['ascend_parcel'] != matches['detrain_parcel']]
    #returns matches, dropping duplicates since two detrainment regions can overlap
    intersections = matches.drop_duplicates(['t', 'ascend_parcel', 'detrain_parcel']).copy()
    intersections['detrain_cloudType'] = GetDetrainCloudType(intersections, isDetrain, cloudTypeMap_Lagrangian)
    return intersections

def CellRadiusKmToIndex(cellRadius_km, dx):
    """Converts a physical search radius (km) to a number of gridboxes, rounding to nearest."""
    return int(round(cellRadius_km / dx))

def GetNeighborCellIds(ix, iy, iz, dims, radius=1):
    """
    Given arrays of grid indices (one per event), returns the cellId of
    every horizontal neighbor within `radius` gridboxes (Z held fixed).
    Uses the same ravel_multi_index encoding as AssignCellId/InverseCellId,
    so the resulting IDs are directly comparable to your existing cellId array.
    """
    # the set of horizontal offsets to check around each point, e.g.
    # radius=1 -> [-1, 0, 1] in both x and y (a 3x3 block, 9 combinations)
    offsetRange = np.arange(-radius, radius + 1)
    offsetX, offsetY = np.meshgrid(offsetRange, offsetRange, indexing='ij')
    offsetX, offsetY = offsetX.ravel(), offsetY.ravel()   # flatten the 3x3 grid into 9 (dx,dy) pairs
    numNeighbors = len(offsetX)                            # 9 for radius=1, 25 for radius=2, etc.

    # for every input point (rows) and every offset (columns), compute the 
    # neighbor's x-index = point's x + offset, then clamp it into [0, dims[0]-1] 
    # so it never points outside the grid at domain edges
    neighborX = np.clip(ix[:, None] + offsetX[None, :], 0, dims[0] - 1)
    neighborY = np.clip(iy[:, None] + offsetY[None, :], 0, dims[1] - 1)
    #Z stays at same level, search is only in the horizontal
    neighborZ = np.repeat(iz[:, None], numNeighbors, axis=1)  
    
    # re-encode each neighbor's (x,y,z) back into a single cellId, using
    # the same bijective mapping as AssignCellId
    neighborCellIds = np.ravel_multi_index((neighborX, neighborY, neighborZ), dims)
    return [neighborCellIds, numNeighbors]

def AssignCellId(dataDictionary):
    """
    Assign unique gridcell IDs 
    using linear bijective index mapping
    """
    dims = GetGridDims()
    X = dataDictionary['X']; Y = dataDictionary['Y']; Z = dataDictionary['Z']
    return np.ravel_multi_index((X, Y, Z), dims)

def InverseCellId(cellId):
    """
    Converts unique gridcell IDs u
    sing linear bijective index mapping 
    back into X, Y, Z coordinates.
    """
    dims = GetGridDims()
    return np.unravel_index(cellId, dims)

def GetGridDims():
    return (ModelData.Nxh + 1, ModelData.Nyh + 1, ModelData.Nzh + 1)

def GetDetrainCloudType(intersections, isDetrain, cloudTypeMap_Lagrangian):
    """
    For each row in `intersections`, recovers the detrain_parcel's actual
    detrainment timestep (lost during FindIntersections' window expansion),
    then looks up its cloud type at the timestep right before detrainment
    (the last timestep it was still cloudy).
    """
    dT, dP = np.where(isDetrain)
    detrainEventsDF = pd.DataFrame({'detrain_parcel': dP, 'detrainT': dT})

    left = intersections.reset_index().rename(columns={'index': 'orig_idx'})
    merged = left.merge(detrainEventsDF, on='detrain_parcel', how='left')

    # only detrainment events that occurred at or before this intersection's time
    merged = merged[merged['detrainT'] <= merged['t']]

    # if a detrain_parcel had multiple detrainment events, take the most
    # recent one at/before the intersection (the one that could plausibly
    # be responsible for this specific match)
    merged = merged.sort_values('detrainT').drop_duplicates(subset='orig_idx', keep='last')
    merged = merged.set_index('orig_idx').sort_index()

    preDetrainT = np.clip(merged['detrainT'].values - 1, 0, None).astype(int)
    detrainParcelArr = merged['detrain_parcel'].values.astype(int)
    cloudType = cloudTypeMap_Lagrangian[preDetrainT, detrainParcelArr]

    cloudTypeSeries = pd.Series(cloudType, index=merged.index, name='detrain_cloudType')
    return cloudTypeSeries.reindex(intersections.index)
    
####################################################################################
#Intersection Visualization Plot One
import matplotlib.patheffects as pe

OUTLINE = [pe.withStroke(linewidth=3, foreground='white')]

def PlotIntersection(dataDictionary, intersections, row=0, dx_km=1.0,
                      xlim=(300, 320), ylim=(0, 5), ax=None, tOverride=None,
                      plotVar='w', plotType='contourf', nContours=15, cellRadius_km=1):
    cfg = VAR_CONFIG[plotVar]

    rec = intersections.iloc[row]
    tCenter = int(rec['t'])
    t = tCenter if tOverride is None else tOverride
    cellId = int(rec['cellId'])
    ascendP = int(rec['ascend_parcel'])
    detrainP = int(rec['detrain_parcel'])
    ix, iy, iz = InverseCellId(cellId)
    Nt = dataDictionary['Z'].shape[0]
    t = max(0, min(t, Nt - 1))
    tStepAbs = dataDictionary['tSteps'][t]

    field = CallVariable(ModelData, DataManager, ModelData.timeStrings[tStepAbs], cfg['varname'])
    fieldSlice = field[:, iy, :] * cfg['scale']

    nz, nx = fieldSlice.shape
    xGrid = np.arange(nx) * dx_km
    zGrid = ModelData.zh[:nz]

    ownFig = ax is None
    if ownFig:
        fig, ax = plt.subplots(figsize=(8, 5))

    if plotType == 'pcolormesh':
        im = ax.pcolormesh(xGrid, zGrid, fieldSlice, cmap=cfg['cmap'],
                            vmin=cfg['vmin'], vmax=cfg['vmax'], shading='auto')
    else:
        levels = np.linspace(cfg['vmin'], cfg['vmax'], nContours)
        im = ax.contourf(xGrid, zGrid, fieldSlice, levels=levels, cmap=cfg['cmap'], extend='both')

    xA = dataDictionary['X'][:t+1, ascendP] * dx_km
    zA = ModelData.zh[dataDictionary['Z'][:t+1, ascendP]]
    ax.plot(xA, zA, color='black', lw=1.5, marker='o', ms=3, label=f'ascending parcel {ascendP}',
            path_effects=OUTLINE)
    ax.scatter(xA[-1], zA[-1], color='lime', s=90, edgecolor='black', zorder=5,
               label='ascending parcel')

    # --- only show the detraining parcel once detrainment has actually happened ---
    dT_events, dP_events = np.where(isDetrain)
    detrainT = dT_events[(dP_events == detrainP)][0]

    if t >= detrainT:
        xD = dataDictionary['X'][:detrainT+1, detrainP] * dx_km
        zD = ModelData.zh[dataDictionary['Z'][:detrainT+1, detrainP]]
        ax.plot(xD, zD, color='blue', lw=1.5, marker='s', ms=3, label=f'detraining parcel {detrainP}',
                path_effects=OUTLINE)
        ax.scatter(xD[-1], zD[-1], color='magenta', s=90, edgecolor='black', zorder=5,
                   label='detrainment location')

        # --- horizontal line showing the cellRadius_km search extent, at the detrainment height ---
        ax.plot([xD[-1] - cellRadius_km, xD[-1] + cellRadius_km], [zD[-1], zD[-1]],
                color='magenta', lw=2, ls='-', alpha=0.7, zorder=4,
                label=f'±{cellRadius_km} km search radius', path_effects=OUTLINE)

    ax.set_xlabel('x (km)')
    ax.set_ylabel('z (km)')
    timeLabel = HoursToHHMM(ModelData.time_hrs[tStepAbs])
    tag = f't={timeLabel}'
    ax.set_title(f'{plotVar} slice at y={iy*dx_km:.1f} km, {tag}')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    if ownFig:
        plt.colorbar(im, ax=ax, label=cfg['label'])
        ax.legend(fontsize=8, loc='best')
        plt.tight_layout()
        plt.show()
    return im

VAR_CONFIG = {
    'w':  {'varname': 'winterp', 'cmap': 'RdBu_r', 'vmin': -5,  'vmax': 5,   'label': 'w (m/s)',
           'scale': 1.0, 'maskBelow': None},
    'qc': {'varname': 'qcqi',    'cmap': 'turbo',  'vmin': 0.01, 'vmax': 2,  'label': 'QCQI (g/kg)',
           'scale': 1e3, 'maskBelow': 0.01},
}

def PlotIntersectionTimeSeries(dataDictionary, intersections, row=0, tOffsets=np.arange(-12,6,2),
                                 ncols=3, plotVar='w', legendPanel=1, **kwargs):
    cfg = VAR_CONFIG[plotVar]
    rec = intersections.iloc[row]
    tCenter = int(rec['t'])
    Nt = dataDictionary['Z'].shape[0]
    tList = [tCenter + off for off in tOffsets if 0 <= tCenter + off < Nt]
    nPanels = len(tList)
    nrows = int(np.ceil(nPanels / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 5*nrows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    im = None
    for ax, t in zip(axes, tList):
        im = PlotIntersection(dataDictionary, intersections, row=row, ax=ax, tOverride=t,
                               plotVar=plotVar, **kwargs)

    for ax in axes[nPanels:]:
        ax.axis('off')

    fig.colorbar(im, ax=axes[:nPanels], label=cfg['label'], shrink=0.6)

    # gather legend entries from ALL panels, drop duplicate labels
    allHandles, allLabels = [], []
    for ax in axes[:nPanels]:
        h, l = ax.get_legend_handles_labels()
        allHandles += h
        allLabels += l
    seen = dict(zip(allLabels, allHandles))

    # place the legend INSIDE a specific panel (row 0, col legendPanel)
    axes[legendPanel].legend(seen.values(), seen.keys(), fontsize=10, loc='best',
                          framealpha=0.85, handlelength=1.5, labelspacing=0.3)

    plt.show()

####################################################################################
#Intersection Visualization Plot Two

PARCEL_VAR_CONFIG = {
    'W':  {'scale': 1.0, 'name': 'W', 'unit': 'm/s'},
    'QV': {'scale': 1e3, 'name': 'QV', 'unit': 'g/kg'},
    'QCQI': {'scale': 1e3, 'name': 'QC+QI', 'unit': 'g/kg'},
}

def PlotParcelProfile(dataDictionary, intersections, row=0, varName='QV', plotDerivative=False):
    cfg = PARCEL_VAR_CONFIG[varName]

    rec = intersections.iloc[row]
    tIntersect = int(rec['t'])
    ascendP = int(rec['ascend_parcel'])
    detrainP = int(rec['detrain_parcel'])
    cellId = int(rec['cellId'])

    ix, iy, iz = InverseCellId(cellId)
    zIntersect = ModelData.zh[iz]

    Z = dataDictionary['Z']
    var = dataDictionary[varName]

    dT_events, dP_events = np.where(isDetrain)
    detrainT = dT_events[(dP_events == detrainP)][0]
    zDetrain = ModelData.zh[Z[detrainT, detrainP]]

    # NEW -- ascent start time/height for the ascending parcel
    ascentStartByParcel = GetAscentStartByParcel(isUpdraftEvent)
    ascentStartT = ascentStartByParcel[ascendP]
    zAscentStart = ModelData.zh[Z[ascentStartT, ascendP]]

    tSteps = dataDictionary['tSteps']
    timeOfDay = ModelData.time_hrs[tSteps]
    timeMinutes = timeOfDay * 60.0

    # full trajectory data for both parcels
    zA = ModelData.zh[Z[:, ascendP]]
    varA = var[:, ascendP] * cfg['scale']

    zD = ModelData.zh[Z[:, detrainP]]
    varD = var[:, detrainP] * cfg['scale']

    if plotDerivative:
        plotVarA = SafeGradient(varA, timeMinutes)
        plotVarD = SafeGradient(varD, timeMinutes)
        label = f"d{cfg['name']}/dt ({cfg['unit']}/min)"
    else:
        plotVarA, plotVarD = varA, varD
        label = f"{cfg['name']} ({cfg['unit']})"

    valAtIntersect = plotVarA[tIntersect]
    valAtAscentStart = plotVarA[ascentStartT]   # NEW

    fig = plt.figure(figsize=(13, 6))
    gs = GridSpec(1, 2, width_ratios=[1, 1.3], wspace=0.15, figure=fig)

    # --- LEFT: height profile ---
    axZ = fig.add_subplot(gs[0, 0])
    axZ.plot(plotVarA, zA, color='black', lw=1.5, marker='o', ms=3,
             label=f'ascending parcel {ascendP}')
    axZ.plot(plotVarD, zD, color='blue', lw=1.5, marker='s', ms=3,
             label=f'detraining parcel {detrainP}')
    axZ.axhline(zIntersect, color='green', ls='--', lw=1.5, label='intersection moment')
    axZ.axhline(zDetrain, color='magenta', ls=':', lw=1.5, label='detrainment moment')
    axZ.scatter(valAtIntersect, zIntersect, color='lime', s=100,lw=1.4, edgecolor='black',
                zorder=5, label='ascending parcel at intersection')
    axZ.scatter(valAtAscentStart, zAscentStart, facecolors='orange', edgecolors='black',   
                s=120, lw=1.4, zorder=5, label='ascending parcel: initial acceleration') 
    if plotDerivative:
        axZ.axvline(0, color='gray', lw=0.8, alpha=0.5)
    axZ.set_xlabel(label,fontsize=12)
    axZ.set_ylabel('z (km)',fontsize=12)

    # --- RIGHT: time profile ---
    axT = fig.add_subplot(gs[0, 1])
    axT.plot(timeOfDay, plotVarA, color='black', lw=1.5, marker='o', ms=3,
             label=f'ascending parcel {ascendP}')
    axT.plot(timeOfDay, plotVarD, color='blue', lw=1.5, marker='s', ms=3,
             label=f'detraining parcel {detrainP}')
    axT.axvline(timeOfDay[detrainT], color='magenta', ls=':', lw=1.5, label='detrainment moment')
    axT.axvline(timeOfDay[tIntersect], color='green', ls='--', lw=1.5, label='intersection moment')
    axT.scatter(timeOfDay[tIntersect], valAtIntersect, color='lime', s=100,lw=1.4,
                edgecolor='black', zorder=5, label='ascending parcel at intersection')
    axT.scatter(timeOfDay[ascentStartT], valAtAscentStart, facecolors='orange', edgecolors='black',
                s=120, lw=1.4, zorder=5, label='ascending parcel: initial acceleration')         
    if plotDerivative:
        axT.axhline(0, color='gray', lw=0.8, alpha=0.5)

    axT.xaxis.set_major_formatter(FuncFormatter(HoursToHHMM))
    axT.set_xlabel('Time (LT)',fontsize=12)
    axT.set_ylabel(label,fontsize=12)
    fig.autofmt_xdate()

    # combined legend from both panels, deduplicated
    handles, labels = [], []
    for ax in (axZ, axT):
        h, l = ax.get_legend_handles_labels()
        handles += h
        labels += l
    seen = dict(zip(labels, handles))
    fig.legend(seen.values(), seen.keys(), fontsize=12, loc='upper center',
               ncol=3, bbox_to_anchor=(0.5, 1.0))

def GetAscentStartByParcel(isUpdraftEvent):
    evT, evP = np.where(isUpdraftEvent)
    return dict(zip(evP.tolist(), evT.tolist()))

def SafeGradient(f, x):
    """
    d(f)/d(x), robust to duplicate/zero-spacing values in x (e.g. a parcel
    staying at the same discrete height level for consecutive timesteps).
    Computes via chain rule (df/di)/(dx/di) instead of np.gradient(f, x)
    directly, and returns NaN where dx==0 rather than raising a divide warning.
    """
    df = np.gradient(f)
    dx = np.gradient(x)
    with np.errstate(divide='ignore', invalid='ignore'):
        deriv = np.where(dx != 0, df / dx, np.nan)
    return deriv

In [ ]:
##############################################
#COMPUTING

In [ ]:
dataDictionary=GetData(tRange=[12,13],pValues=np.arange(20_000_000))
cloudTypeMap_Lagrangian=CalculateLagrangianCloudType(dataDictionary,tRange=[12,13])

In [ ]:
#Track Ascending Surface Parcels
isUpdraftEvent = DetectUpdraftEvents(dataDictionary)
isAscending = GetAscentDurationMask(dataDictionary, isUpdraftEvent)

# #Visualization
# examples = np.where(isAscending)[1]
# PlotAscentValidation(dataDictionary, isAscending, p=examples[5])

In [ ]:
#Find Detraining Parcels at MidLevels
isDetrain=DetectDetrainment(dataDictionary)

In [ ]:
#Find Ascending-Parcel Intersections with Detrainment Events
intersections = FindIntersections(dataDictionary, isAscending, isDetrain)
print(intersections)

# # #Visualization
# # PlotIntersectionTimeSeries(dataDictionary, intersections, row=1, plotVar='w',
# #                            xlim=(355, 365))
# # PlotIntersectionTimeSeries(dataDictionary, intersections, row=1, plotVar='qc',
# #                            xlim=(355, 365))

# # #Visualization
# PlotIntersectionTimeSeries(dataDictionary, intersections, row=9, plotVar='w',
#                            xlim=(200, 210))
# PlotIntersectionTimeSeries(dataDictionary, intersections, row=9, plotVar='qc',
#                            xlim=(200, 210))
# PlotParcelProfile(dataDictionary, intersections, row=9, varName='W', plotDerivative=False)
# PlotParcelProfile(dataDictionary, intersections, row=9, varName='W', plotDerivative=True)
# PlotParcelProfile(dataDictionary, intersections, row=9, varName='QV', plotDerivative=True)
# PlotParcelProfile(dataDictionary, intersections, row=9, varName='QCQI', plotDerivative=False)

In [ ]:
##############################################
#PLOTTING FUNCTIONS

In [ ]:
#Computing DCI Probabilities

def ComputeDCIProbabilitiesByCloudType(dataDictionary, isAscending, intersections, 
                                       cloudTypeLabels={1:'Cumulus', 2:'Congestus', 3:'Cumulonimbus', 4:'Overshooting'}):
    """
    Runs ComputeDeepConvectionProbability + PlotProbabilityVsHeightThreshold
    once on the full (unfiltered) intersections, then once per cloud type,
    filtering intersections to only that detraining cloud type each time.
    Returns a dict keyed by label, holding each run's results/plot data.
    """
    if cloudTypeLabels is None:
        cloudTypeLabels = {1: 'Cumulus', 2: 'Congestus', 3: 'Cumulonimbus'}

    subsets = [('All', intersections)]
    for ct, label in cloudTypeLabels.items():
        subsets.append((label, intersections[intersections['detrain_cloudType'] == ct]))

    DCI_Probabilities = {}
    for label, subsetIntersections in subsets:
        print(f"\n========== {label} (n intersections = {len(subsetIntersections)}) ==========")
        results = ComputeDeepConvectionProbability(dataDictionary, isAscending, subsetIntersections)
        zThresholds, pIntersected, pNotIntersected = PlotProbabilityVsHeightThreshold(
            dataDictionary, isAscending, subsetIntersections
        )
        DCI_Probabilities[label] = {
            'results': results, 'zThresholds': zThresholds,
            'pIntersected': pIntersected, 'pNotIntersected': pNotIntersected
        }
    return DCI_Probabilities

def ComputeDeepConvectionProbability(dataDictionary, isAscending, intersections,
                                      deepThresh=6.0, shallowThresh=4.0):
    """
    deepThresh: height (km) above which a parcel counts as having gone deep
    shallowThresh: height (km) below which a parcel counts as having stayed shallow

    Only counts height reached AFTER each parcel's intersection time (for
    intersected parcels) or after its ascent start (for non-intersected
    parcels), so "deep" reflects what happened following the reference
    event, not a peak that occurred before it.
    """
    Z = dataDictionary['Z']
    zLevel = ModelData.zh[Z]              # (Nt, Np)
    Nt = zLevel.shape[0]

    allAscendingParcels = np.unique(np.where(isAscending)[1])

    # earliest intersection time per parcel (a parcel could intersect more
    # than once -- use its first, since that's the earliest point causality
    # could plausibly apply)
    firstIntersectTimeByParcel = intersections.groupby('ascend_parcel')['t'].min().to_dict()
    intersectedParcels = np.array(list(firstIntersectTimeByParcel.keys()))
    notIntersectedParcels = np.setdiff1d(allAscendingParcels, intersectedParcels)

    ascentStartByParcel = GetAscentStartByParcel(isUpdraftEvent)

    def TopHeightAfter(parcelIds, refTimeByParcel):
        """Max height reached by each parcel, restricted to timesteps at/after its reference time."""
        heights = np.full(len(parcelIds), np.nan)
        for i, p in enumerate(parcelIds):
            t0 = refTimeByParcel.get(p)
            if t0 is None:
                continue
            heights[i] = zLevel[t0:, p].max()
        return heights

    topHeightIntersected = TopHeightAfter(intersectedParcels, firstIntersectTimeByParcel)
    topHeightNotIntersected = TopHeightAfter(notIntersectedParcels, ascentStartByParcel)

    def ClassFraction(heights, thresh, above=True):
        valid = ~np.isnan(heights)
        heights = heights[valid]
        if len(heights) == 0:
            return np.nan, 0, 0
        isMatch = (heights >= thresh) if above else (heights < thresh)
        return isMatch.mean(), isMatch.sum(), len(heights)

    pDeepGivenIntersected, nDeepIntersected, nIntersected = ClassFraction(topHeightIntersected, deepThresh, above=True)
    pDeepGivenNotIntersected, nDeepNotIntersected, nNotIntersected = ClassFraction(topHeightNotIntersected, deepThresh, above=True)

    pShallowGivenIntersected, nShallowIntersected, _ = ClassFraction(topHeightIntersected, shallowThresh, above=False)
    pShallowGivenNotIntersected, nShallowNotIntersected, _ = ClassFraction(topHeightNotIntersected, shallowThresh, above=False)

    print("--- Deep convection AFTER Intersection time (top height >= {:.1f} km) ---".format(deepThresh))
    print(f"Intersected:     {nDeepIntersected} of {nIntersected} parcels went deep, P(Deep|Intersected) = {pDeepGivenIntersected*100:.2f}%")
    print(f"Not Intersected: {nDeepNotIntersected} of {nNotIntersected} parcels went deep, P(Deep|Not Intersected) = {pDeepGivenNotIntersected*100:.2f}%")

    nDeepTotal = nDeepIntersected + nDeepNotIntersected
    fracDeepThatWereIntersected = nDeepIntersected / nDeepTotal if nDeepTotal > 0 else np.nan
    print(f"Of {nDeepTotal} total deep parcels, {nDeepIntersected} ({fracDeepThatWereIntersected*100:.1f}%) "
          f"intersected a detrainment event before going deep")

    print("\n--- Stayed shallow AFTER Intersection Time (top height < {:.1f} km) ---".format(shallowThresh))
    print(f"Intersected:     {nShallowIntersected} of {nIntersected} parcels stayed shallow, P(Shallow|Intersected) = {pShallowGivenIntersected*100:.2f}%")
    print(f"Not Intersected: {nShallowNotIntersected} of {nNotIntersected} parcels stayed shallow, P(Shallow|Not Intersected) = {pShallowGivenNotIntersected*100:.2f}%")

    return {
        'pDeepGivenIntersected': pDeepGivenIntersected, 'pDeepGivenNotIntersected': pDeepGivenNotIntersected,
        'pShallowGivenIntersected': pShallowGivenIntersected, 'pShallowGivenNotIntersected': pShallowGivenNotIntersected,
        'nIntersected': nIntersected, 'nNotIntersected': nNotIntersected,
    }

def PlotProbabilityVsHeightThreshold(dataDictionary, isAscending, intersections,
                                       zRange=(1, 16), nSteps=40):
    Z = dataDictionary['Z']
    zLevel = ModelData.zh[Z]
    topHeight = zLevel.max(axis=0)

    allAscendingParcels = np.unique(np.where(isAscending)[1])
    intersectedParcels = intersections['ascend_parcel'].unique()
    notIntersectedParcels = np.setdiff1d(allAscendingParcels, intersectedParcels)

    zThresholds = np.linspace(zRange[0], zRange[1], nSteps)

    # vectorized: for every threshold, fraction of each group with topHeight >= threshold
    topIntersected = topHeight[intersectedParcels]
    topNotIntersected = topHeight[notIntersectedParcels]

    pIntersected = np.array([(topIntersected >= z).mean() for z in zThresholds])
    pNotIntersected = np.array([(topNotIntersected >= z).mean() for z in zThresholds])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(zThresholds, pIntersected * 100, color='tab:orange', lw=2, label='Intersected')
    ax.plot(zThresholds, pNotIntersected * 100, color='tab:blue', lw=2, label='Not Intersected')
    ax.set_xlabel('height threshold, z_top (km)')
    ax.set_ylabel('P(parcel reaches z_top) (%)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return zThresholds, pIntersected, pNotIntersected

In [ ]:
# PlotIntersectingHistograms
############################################################################
def PlotIntersectingHistogramsByCloudType(dataDictionary, intersections, varNames=['QV', 'W'],
                                               cloudTypeLabels=None, **kwargs):
    """
    Runs PlotIntersectingHistograms once on the full (unfiltered)
    intersections, then once per cloud type, filtering intersections to
    only that detraining cloud type each time. **kwargs are passed through
    to PlotIntersectingHistograms (windowSteps, deepThresh, normalize,
    plotDerivative, etc.)
    """
    if cloudTypeLabels is None:
        cloudTypeLabels = {1: 'Cumulus', 2: 'Congestus', 3: 'Cumulonimbus'}

    subsets = [('All', intersections)]
    for ct, label in cloudTypeLabels.items():
        subsets.append((label, intersections[intersections['detrain_cloudType'] == ct]))

    histDF_by_type = {}
    for label, subsetIntersections in subsets:
        print(f"\n========== {label} (n intersections = {len(subsetIntersections)}) ==========")
        if len(subsetIntersections) == 0:
            print("(skipped -- no intersections for this cloud type)")
            continue
        histDF = PlotIntersectingHistograms(dataDictionary, subsetIntersections, varNames=varNames, **kwargs)
        histDF_by_type[label] = histDF

    return histDF_by_type
    
def PlotIntersectingHistograms(dataDictionary, intersections, varNames=['QV', 'W'],
                                    windowSteps=10, deepThresh=6.0, dt=None, normalize=True,
                                    plotDerivative=False):
    firstIntersectTimeByParcel = intersections.groupby('ascend_parcel')['t'].min().to_dict()
    isDeep_byParcel = IsDeepAfterReference(dataDictionary, firstIntersectTimeByParcel, deepThresh)

    histDF = BuildIntersectingHistograms(dataDictionary, intersections,
                                             isDeep_byParcel, varNames, windowSteps)
    dt = ModelData.dt if dt is None else dt
    histDF['minutes'] = histDF['offset'] * dt / 60

    if plotDerivative:
        histDF = AddDerivativeColumns(histDF, varNames, dt)
        plotVarNames = [f'd{v}_dt' for v in varNames]
        varLabels = {f'd{v}_dt': f'd{v}/dt (per min)' for v in varNames}
    else:
        plotVarNames = varNames
        varLabels = {v: v for v in varNames}

    outcomeColumns = ['all', 'shallow', 'deep']   # NEW -- 'all' column added, leftmost

    nRows = len(plotVarNames)
    fig, axes = plt.subplots(nRows, 3, figsize=(15, 4.5*nRows))   # CHANGED: 2 -> 3 columns
    axes = np.atleast_2d(axes)
    plt.subplots_adjust(wspace=0.3, hspace=0.3, right=0.9)        # CHANGED: right nudged slightly for 3-col layout

    for i, var in enumerate(plotVarNames):
        cfg = HIST_VAR_CONFIG.get(var, {'nBinsY': 40, 'maskBelow': None, 'yscale': 'linear'})

        varVals = histDF.loc[histDF['outcome'].isin(['shallow', 'deep']), var].copy()
        varVals = varVals.dropna()
        if cfg['maskBelow'] is not None:
            varVals = varVals[varVals >= cfg['maskBelow']]
        if len(varVals) == 0:
            continue
        yMin, yMax = varVals.min(), varVals.max()
        yBins = (np.logspace(np.log10(yMin), np.log10(yMax), cfg['nBinsY'])
                 if cfg['yscale'] == 'log' else
                 np.linspace(yMin, yMax, cfg['nBinsY']))

        rowData = {}
        for outcome in outcomeColumns:   # CHANGED: loop now includes 'all'
            if outcome == 'all':
                sub = histDF.dropna(subset=[var]).copy()   # NEW -- both shallow and deep combined, no outcome filter
            else:
                sub = histDF[histDF['outcome'] == outcome].dropna(subset=[var]).copy()
            if cfg['maskBelow'] is not None:
                sub = sub[sub[var] >= cfg['maskBelow']]
            if len(sub) == 0:
                rowData[outcome] = None
                continue
            xBins = np.linspace(sub['minutes'].min(), sub['minutes'].max(), 2*windowSteps+1)
            counts, xEdges, yEdges = np.histogram2d(sub['minutes'], sub[var], bins=[xBins, yBins])
            if normalize:
                colSums = counts.sum(axis=1, keepdims=True)
                colSums[colSums == 0] = 1
                plotData = counts / colSums
            else:
                plotData = counts
            rowData[outcome] = {'data': plotData, 'xEdges': xEdges, 'yEdges': yEdges,
                                 'n': sub['parcel'].nunique()}

        validVmax = [rowData[o]['data'].max() for o in outcomeColumns if rowData[o] is not None]   # CHANGED: outcomeColumns
        vmax = max(validVmax) if validVmax else 1
        cbarLabel = 'fraction (per time-column)' if normalize else 'count'

        im = None
        for j, outcome in enumerate(outcomeColumns):   # CHANGED: outcomeColumns
            ax = axes[i, j]
            d = rowData[outcome]
            if d is None:
                ax.set_title(f'{var} — {outcome} (no data)')
                continue
            im = ax.pcolormesh(d['xEdges'], d['yEdges'], d['data'].T, cmap='viridis',
                                shading='auto', vmin=0, vmax=vmax)
            ax.axvline(0, color='white', ls='--', lw=1)
            ax.axhline(0, color='gray', lw=0.8, alpha=0.5) if plotDerivative else None
            ax.set_xlabel('minutes since intersection')
            ax.set_ylabel(varLabels[var])
            if cfg['yscale'] == 'log':
                ax.set_yscale('log')
            ax.set_title(f'{varLabels[var]} — {outcome} (n={d["n"]} parcels)')

        if im is not None:
            pos = axes[i, 2].get_position()   # CHANGED: colorbar now anchored to column 2 (deep), the new rightmost panel
            gap = 0.01
            width = 0.015
            cax = fig.add_axes([pos.x1 + gap, pos.y0, width, pos.height])
            fig.colorbar(im, cax=cax, label=cbarLabel)

    plt.show()
    return histDF

def BuildIntersectingHistograms(dataDictionary, intersections, isDeep_byParcel, varNames=['QV','W'], windowSteps=10):
    """
    For each intersection event, pull the ascending parcel's variable values
    for `windowSteps` timesteps before/after the intersection, tagged by
    whether that parcel went deep AFTER that intersection.
    """
    records = []
    for _, row in intersections.iterrows():
        p = int(row['ascend_parcel'])
        t0 = int(row['t'])
        if p not in isDeep_byParcel:
            continue
        outcome = 'deep' if isDeep_byParcel[p] else 'shallow'
        for offset in range(-windowSteps, windowSteps+1):
            t = t0 + offset
            if 0 <= t < dataDictionary['Z'].shape[0]:
                rec = {'parcel': p, 'offset': offset, 'outcome': outcome}
                for v in varNames:
                    rec[v] = dataDictionary[v][t, p]
                records.append(rec)
    return pd.DataFrame(records)

def AddDerivativeColumns(df, varNames, dt):
    """
    For each parcel, computes d(var)/dt (per minute) across its own set of
    (offset, var) rows in the long-format histogram dataframe, and adds
    new columns named 'd{var}_dt'.
    """
    df = df.sort_values(['parcel', 'offset']).copy()
    for var in varNames:
        derivCol = f'd{var}_dt'
        df[derivCol] = np.nan
        for p, group in df.groupby('parcel'):
            idx = group.index
            timeMin = group['offset'].values * dt / 60.0
            values = group[var].values
            df.loc[idx, derivCol] = SafeGradient(values, timeMin)
    return df


def GetTopHeightAfterReference(dataDictionary, refTimeByParcel):
    """
    For each parcel with a known reference time t0, returns the max height
    reached from t0 onward (not the whole loaded window). refTimeByParcel
    is a dict: parcel_id -> t0.
    """
    Z = dataDictionary['Z']
    zLevel = ModelData.zh[Z]
    topHeight = {}
    for p, t0 in refTimeByParcel.items():
        topHeight[p] = zLevel[t0:, p].max()
    return topHeight

def IsDeepAfterReference(dataDictionary, refTimeByParcel, deepThresh=6.0):
    """
    dict: parcel_id -> bool, True if the parcel reached deepThresh at or
    after its own reference time (intersection time, or ascent start for
    parcels with no intersection). Only covers parcels present in
    refTimeByParcel.
    """
    topHeight = GetTopHeightAfterReference(dataDictionary, refTimeByParcel)
    return {p: (h >= deepThresh) for p, h in topHeight.items()}

In [ ]:
# PlotComparisonHistograms: variable evolution around ascent start,
# Intersecting vs. truly-non-Intersecting parcels, split by shallow/deep outcome
############################################################################

HIST_VAR_CONFIG = {
    'W':    {'nBinsY': 40, 'maskBelow': None, 'yscale': 'linear'},
    'QV':   {'nBinsY': 40, 'maskBelow': None, 'yscale': 'linear'},
    'QCQI': {'nBinsY': 60, 'maskBelow': 1e-6, 'yscale': 'log'},
    'RH_vapor': {'nBinsY': 60, 'maskBelow': None, 'yscale': 'linear'},
}

def PlotComparisonHistograms(dataDictionary, isAscending, isUpdraftEvent, intersections,
                               allIntersections=None, varNames=['QV','W'], windowSteps=10,
                               deepThresh=6.0, dt=None, normalize=True):
    """
    intersections: the (possibly cloud-type-filtered) subset defining the
        "intersected" group.
    allIntersections: the FULL, unfiltered intersections table, used only
        to determine which parcels intersected NOTHING AT ALL. Defaults to
        `intersections` itself (so calling with just `intersections` alone
        reproduces the old "intersected vs. everyone else in this subset"
        behavior). Parcels that intersected something, but not the specific
        type being studied here, are excluded from both groups entirely.
    """
    if allIntersections is None:
        allIntersections = intersections

    ascentStartByParcel = GetAscentStartByParcel(isUpdraftEvent)

    allAscendingParcels = np.unique(np.where(isAscending)[1])
    intersectedParcels = intersections['ascend_parcel'].unique()                    # FIXED: consistent name
    everIntersectedParcels = allIntersections['ascend_parcel'].unique()             # NEW -- intersected ANYTHING
    notIntersectedParcels = np.setdiff1d(allAscendingParcels, everIntersectedParcels)  # FIXED: true "nothing at all"

    firstIntersectTimeByParcel = intersections.groupby('ascend_parcel')['t'].min().to_dict()
    refTimeByParcel = dict(firstIntersectTimeByParcel)
    for p in notIntersectedParcels:
        if p in ascentStartByParcel:
            refTimeByParcel[p] = ascentStartByParcel[p]

    isDeep_byParcel = IsDeepAfterReference(dataDictionary, refTimeByParcel, deepThresh)

    histIntersected = BuildHistogramsFromAscentStart(dataDictionary, intersectedParcels,
                                                       ascentStartByParcel, isDeep_byParcel, varNames, windowSteps)
    histIntersected['group'] = 'intersected'

    histNotIntersected = BuildHistogramsFromAscentStart(dataDictionary, notIntersectedParcels,
                                                          ascentStartByParcel, isDeep_byParcel, varNames, windowSteps)
    histNotIntersected['group'] = 'not intersected'

    histAll = pd.concat([histIntersected, histNotIntersected], ignore_index=True)
    dt = ModelData.dt if dt is None else dt
    histAll['minutes'] = histAll['offset'] * dt / 60

    interTimes = intersections.merge(
        pd.Series(ascentStartByParcel, name='ascentStart').rename_axis('ascend_parcel').reset_index(),
        on='ascend_parcel', how='left'
    )
    interTimes['offsetFromAscentStart_min'] = (interTimes['t'] - interTimes['ascentStart']) * dt / 60
    meanInterOffset = interTimes['offsetFromAscentStart_min'].mean()

    outcomeColumns = ['all', 'shallow', 'deep']
    rowOrder = ['not intersected', 'intersected']
    groupOutcomePairs = [(grp, outcome) for grp in rowOrder for outcome in outcomeColumns]

    for var in varNames:
        cfg = HIST_VAR_CONFIG.get(var, {'nBinsY': 40, 'maskBelow': None, 'yscale': 'linear'})
        varVals = histAll[var]
        if cfg['maskBelow'] is not None:
            varVals = varVals[varVals >= cfg['maskBelow']]
        if len(varVals) == 0:
            continue
        yMin, yMax = varVals.min(), varVals.max()
        yBins = (np.logspace(np.log10(yMin), np.log10(yMax), cfg['nBinsY'])
                 if cfg['yscale']=='log' else np.linspace(yMin, yMax, cfg['nBinsY']))

        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        plt.subplots_adjust(wspace=0.3, hspace=0.4, right=0.9)

        cellData = {}
        for grp, outcome in groupOutcomePairs:
            if outcome == 'all':
                sub = histAll[histAll['group']==grp].copy()
            else:
                sub = histAll[(histAll['group']==grp) & (histAll['outcome']==outcome)].copy()
            if cfg['maskBelow'] is not None:
                sub = sub[sub[var] >= cfg['maskBelow']]
            if len(sub) == 0:
                cellData[(grp,outcome)] = None
                continue
            xBins = np.linspace(sub['minutes'].min(), sub['minutes'].max(), 2*windowSteps+1)
            counts, xEdges, yEdges = np.histogram2d(sub['minutes'], sub[var], bins=[xBins, yBins])
            if normalize:
                colSums = counts.sum(axis=1, keepdims=True)
                colSums[colSums == 0] = 1
                plotData = counts / colSums
            else:
                plotData = counts
            cellData[(grp,outcome)] = {'data': plotData, 'xEdges': xEdges, 'yEdges': yEdges,
                                        'n': sub['parcel'].nunique()}

        validVmax = [d['data'].max() for d in cellData.values() if d is not None]
        vmax = max(validVmax) if validVmax else 1
        cbarLabel = 'fraction (per time-column)' if normalize else 'count'

        im = None
        for ax, (grp, outcome) in zip(axes.ravel(), groupOutcomePairs):
            d = cellData[(grp,outcome)]
            if d is None:
                ax.set_title(f'{grp} / {outcome} (no data)')
                continue
            im = ax.pcolormesh(d['xEdges'], d['yEdges'], d['data'].T, cmap='viridis',
                                shading='auto', vmin=0, vmax=vmax)
            ax.axvline(0, color='white', ls='--', lw=1, label='ascent start')
            if grp == 'intersected':
                ax.axvline(meanInterOffset, color='red', ls=':', lw=1.5, label='mean intersection time')
                ax.legend(fontsize=7, loc='upper right')
            if cfg['yscale']=='log': ax.set_yscale('log')
            ax.set_xlabel('minutes since ascent start', fontsize=10)
            ax.set_ylabel(var, fontsize=10)
            ax.set_title(f'{grp} / {outcome} (n={d["n"]})', fontsize=10)
            ax.tick_params(labelsize=9)

        if im is not None:
            fig.colorbar(im, ax=axes.ravel().tolist(), label=cbarLabel, shrink=0.7)
            cbar_ax = fig.axes[-1]
            cbar_ax.set_ylabel(cbarLabel, fontsize=10)
            cbar_ax.tick_params(labelsize=9)

        fig.suptitle(f'{var} — intersected vs. not intersected at all, all/shallow/deep', fontsize=13)
        plt.show()

    return histAll


def GetAscentStartByParcel(isUpdraftEvent):
    evT, evP = np.where(isUpdraftEvent)
    return dict(zip(evP.tolist(), evT.tolist()))


def BuildHistogramsFromAscentStart(dataDictionary, parcelIds, ascentStartByParcel,
                                     isDeep_byParcel, varNames=['QV','W'], windowSteps=10):
    records = []
    Nt = dataDictionary['Z'].shape[0]
    for p in parcelIds:
        if p not in ascentStartByParcel or p not in isDeep_byParcel:
            continue
        t0 = ascentStartByParcel[p]
        outcome = 'deep' if isDeep_byParcel[p] else 'shallow'
        for offset in range(-windowSteps, windowSteps+1):
            t = t0 + offset
            if 0 <= t < Nt:
                rec = {'parcel': p, 'offset': offset, 'outcome': outcome}
                for v in varNames:
                    rec[v] = dataDictionary[v][t, p]
                records.append(rec)
    return pd.DataFrame(records)


#Plot Comparison Histograms
def PlotComparisonHistogramsByCloudType(dataDictionary, isAscending, isUpdraftEvent, intersections,
                                          varNames=['QV', 'W'], cloudTypeLabels=None, **kwargs):
    """
    Runs PlotComparisonHistograms once on the full (unfiltered) intersections,
    then once per cloud type. `allIntersections` (the full, unfiltered table)
    is passed to every call so "not intersected" always means "intersected
    nothing at all" -- parcels preconditioned by a DIFFERENT cloud type are
    excluded from that cloud type's comparison, not counted as a control.
    """
    if cloudTypeLabels is None:
        cloudTypeLabels = {1: 'Cumulus', 2: 'Congestus', 3: 'Cumulonimbus'}

    subsets = [('All', intersections)]
    for ct, label in cloudTypeLabels.items():
        subsets.append((label, intersections[intersections['detrain_cloudType'] == ct]))

    histAll_by_type = {}
    for label, subsetIntersections in subsets:
        print(f"\n========== {label} (n intersections = {len(subsetIntersections)}) ==========")
        if len(subsetIntersections) == 0:
            print("(skipped -- no intersections for this cloud type)")
            continue
        histAll = PlotComparisonHistograms(dataDictionary, isAscending, isUpdraftEvent,
                                            subsetIntersections, allIntersections=intersections,   # NEW
                                            varNames=varNames, **kwargs)
        histAll_by_type[label] = histAll

    return histAll_by_type

In [ ]:
##############################################
#PLOTTING

In [ ]:
#Computing DCI Probabilities
DCI_Probabilities = ComputeDCIProbabilitiesByCloudType(dataDictionary, isAscending, intersections)

In [ ]:
# Plot Property Histograms Relative to Intersection Time
histDF_by_type = PlotIntersectingHistogramsByCloudType(dataDictionary, intersections, varNames=['W', 'QV'])

In [ ]:
# Plot Property Histograms Relative to Intersection Time
histAll_by_type = PlotComparisonHistogramsByCloudType(dataDictionary, isAscending, isUpdraftEvent, intersections, varNames=['W','QV'])